In [11]:
%load_ext autoreload
%autoreload 2

import os, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from gllvm.simulations import make_sparse, simulate
from gllvm.encoder import MapEncoderGaussianLog1p
from gllvm.fitter import ZQEPoissonFitter, ZQELBFGSFitter
from gllvm.glms import PoissonLog1pGLM

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cuda


## Simulation

In [12]:
torch.manual_seed(12)

NL  = 2       # latent dims
ACT = NL      # active latents per response
NR  = 20      # responses
NS  = 100     # observations
WZS = 1.0     # loading scale
RPL = NR // 2 # non-zero loadings per column (sparse)

g0m = make_sparse(n_latent=NL, poisson=NR, active_latent=ACT,
                  wz_scale=WZS, responses_per_latent=RPL).to(DEV)
y_all, z_all = simulate(g0m, n_samples=NS, device=DEV)

print(f"g0m: p={g0m.p}  q={NL}  N={NS}")
print(f"y range: [{y_all.min().item():.0f}, {y_all.max().item():.0f}]")
W_nz = (g0m.wz.detach().cpu() != 0)
print(f"W non-zero per column: {W_nz.sum(0).tolist()}  (of {NR})")


# ── Procrustes helpers ──────────────────────────────────────────────────────
def _best_rotation(A, B):
    U, _, Vt = np.linalg.svd(A.T @ B)
    R1 = (U @ Vt).T
    D  = np.eye(U.shape[0]); D[-1, -1] = -1.0
    R2 = (U @ D @ Vt).T
    return R1 if np.linalg.norm(A - B @ R1) <= np.linalg.norm(A - B @ R2) else R2

def procr(g_true, g_est):
    a = g_true.wz.detach().cpu().numpy()
    b = g_est.wz.detach().cpu().numpy()
    R = _best_rotation(a, b)
    return float(np.linalg.norm(a - b @ R) / np.linalg.norm(a))

g0m: p=20  q=2  N=100
y range: [0, 7307]
W non-zero per column: [10, 10]  (of 20)


## Decoder factory

In [13]:
def fresh_decoder():
    """Fresh dense GLLVM with PoissonLog1pGLM decoder."""
    from gllvm.gllvm_module import GLLVM
    g = GLLVM(latent_dim=NL, output_dim=NR, bias=True).to(DEV)
    g.add_glm(PoissonLog1pGLM, idx=list(range(NR)), name="P")
    with torch.no_grad():
        nn.init.normal_(g.wz, std=WZS)
        nn.init.zeros_(g.bias)
    return g

## Adam ZQE baseline

In [14]:
EPOCHS_ADAM = 1500

torch.manual_seed(SEED)
g_adam = fresh_decoder()
t0 = time.perf_counter()
ft_adam = ZQEPoissonFitter(g_adam, device=DEV)
ft_adam.fit(y_all, epochs=EPOCHS_ADAM)
t_adam = time.perf_counter() - t0

err_adam = procr(g0m, ft_adam.model)
print(f"\nAdam ZQE  Procrustes={err_adam:.4f}  time={t_adam:.1f}s")

    ↓ lr 5.00e-01→2.50e-01  gnorm=1.3084  ep=121
    ↓ lr 2.50e-01→1.25e-01  gnorm=0.7934  ep=187
    ↓ lr 2.50e-01→1.25e-01  gnorm=0.7934  ep=187
    ↓ lr 1.25e-01→6.25e-02  gnorm=0.6107  ep=255
    ↓ lr 1.25e-01→6.25e-02  gnorm=0.6107  ep=255
  ep  300/1500  loss=-0.1512  gnorm=0.8640  lr=6.25e-02
  ep  300/1500  loss=-0.1512  gnorm=0.8640  lr=6.25e-02
    ↓ lr 6.25e-02→3.12e-02  gnorm=1.1903  ep=310
    ↓ lr 6.25e-02→3.12e-02  gnorm=1.1903  ep=310
    ↓ lr 3.12e-02→1.56e-02  gnorm=0.5506  ep=361
    ↓ lr 3.12e-02→1.56e-02  gnorm=0.5506  ep=361
    ↓ lr 1.56e-02→7.81e-03  gnorm=0.6232  ep=412
    ↓ lr 1.56e-02→7.81e-03  gnorm=0.6232  ep=412
    ↓ lr 7.81e-03→3.91e-03  gnorm=0.9298  ep=463
    ↓ lr 7.81e-03→3.91e-03  gnorm=0.9298  ep=463
  ep  600/1500  loss=-1.7219  gnorm=0.5659  lr=3.91e-03
    ↓ lr 3.91e-03→1.95e-03  gnorm=0.4156  ep=605
  ep  600/1500  loss=-1.7219  gnorm=0.5659  lr=3.91e-03
    ↓ lr 3.91e-03→1.95e-03  gnorm=0.4156  ep=605
    ↓ lr 1.95e-03→9.77e-04  gnorm=1.0336 

## L-BFGS EM ZQE

In [15]:
N_OUTER   = 200
SIM_FACTOR = 10          # n_sim = SIM_FACTOR * N  (theory: need ~100*N for 1% efficiency cost)
N_SIM     = SIM_FACTOR * NS

torch.manual_seed(SEED)
g_lbfgs = fresh_decoder()
t0 = time.perf_counter()
ft_lbfgs = ZQELBFGSFitter(g_lbfgs, device=DEV, n_sim=N_SIM)
ft_lbfgs.fit(y_all, n_outer=N_OUTER)
t_lbfgs = time.perf_counter() - t0

err_lbfgs = procr(g0m, ft_lbfgs.model)
print(f"\nL-BFGS ZQE  n_sim={N_SIM} ({SIM_FACTOR}×N)  Procrustes={err_lbfgs:.4f}  time={t_lbfgs:.1f}s")


ValueError: Expected parameter rate (Tensor of shape (1000, 20)) of distribution PoissonLog1pGLM(linpar=tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0')) to satisfy the constraint GreaterThanEq(lower_bound=0.0), but found invalid values:
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0')

## R gllvm VA benchmark

In [ ]:
import subprocess, textwrap
import pandas as pd

RSCRIPT    = "/mnt/c/Program Files/R/R-4.5.1/bin/Rscript.exe"
WIN_TMPDIR = "/mnt/c/Users/willwhite/AppData/Local/Temp/r_gllvm_bench"
os.makedirs(WIN_TMPDIR, exist_ok=True)

def wsl2r(path):
    return path.replace("/mnt/c/", "C:/")

def run_r_gllvm(Y_np, W_true_np, n_latent, method="VA", seed=42):
    y_path = os.path.join(WIN_TMPDIR, "Y.csv")
    w_path = os.path.join(WIN_TMPDIR, f"W_hat_{method}.csv")
    r_path = os.path.join(WIN_TMPDIR, f"run_{method}.R")
    if os.path.exists(w_path):
        os.remove(w_path)
    pd.DataFrame(Y_np).to_csv(y_path, index=False, header=False)
    r_code = textwrap.dedent(f"""
        set.seed({seed})
        Y <- as.matrix(read.csv("{wsl2r(y_path)}", header=FALSE))
        suppressPackageStartupMessages(library(gllvm))
        fit <- gllvm(Y, num.lv={n_latent}, family="poisson",
                     method="{method}", starting.val="res",
                     control=list(maxit=2000), trace=FALSE)
        W_full <- sweep(fit$params$theta, 2, fit$params$sigma.lv, "*")
        write.csv(W_full, "{wsl2r(w_path)}", row.names=FALSE)
        cat("converged\\n")
    """)
    with open(r_path, "w") as f:
        f.write(r_code)
    result = subprocess.run(
        [RSCRIPT, "--vanilla", wsl2r(r_path)],
        capture_output=True, text=True, timeout=600,
    )
    if result.returncode != 0 or not os.path.exists(w_path):
        print("R stderr:", result.stderr[-1500:])
        return None
    W_hat = pd.read_csv(w_path).values
    U, _, Vt = np.linalg.svd(W_true_np.T @ W_hat)
    R = (U @ Vt).T
    err = float(np.linalg.norm(W_true_np - W_hat @ R) / np.linalg.norm(W_true_np))
    return err, W_hat

Y_int    = y_all.cpu().numpy().astype(int)
W_true_r = g0m.wz.detach().cpu().numpy()

print("Fitting R gllvm (VA) ...")
out = run_r_gllvm(Y_int, W_true_r, n_latent=NL)
if out is not None:
    err_r, W_hat_r = out
    print(f"  R gllvm VA  Procrustes = {err_r:.4f}")
else:
    err_r, W_hat_r = float("nan"), None
    print("  R gllvm VA failed")

## Results

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'Method':<40} {'Procrustes':>12} {'Time (s)':>10}")
print("-" * 64)
print(f"{'ZQE Adam (Gaussian MAP, log1p)':<40} {err_adam:>12.4f} {t_adam:>10.1f}")
print(f"{'ZQE L-BFGS EM (Gaussian MAP, log1p)':<40} {err_lbfgs:>12.4f} {t_lbfgs:>10.1f}")
print(f"{'R gllvm VA':<40} {err_r:>12.4f} {'—':>10}")

# ── Training curves ───────────────────────────────────────────────────────────
def smooth(x, w):
    return np.convolve(x, np.ones(w) / w, mode="valid")

SMOOTH = 5
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle(f"ZQE training  (N={NS}, NL={NL}, NR={NR})", fontsize=12)

ax_loss_adam, ax_gn_adam, ax_loss_lbfgs, ax_gn_lbfgs = axes.ravel()

# Adam
ep_a  = np.arange(len(ft_adam.h_loss))
eps_a = np.arange(SMOOTH - 1, len(ft_adam.h_loss))
ax_loss_adam.plot(eps_a, smooth(ft_adam.h_loss,  SMOOTH), color="tab:blue", lw=1.4)
ax_loss_adam.axhline(0, color="gray", ls=":", lw=0.8)
ax_loss_adam.set_title("Adam — ZQE loss");  ax_loss_adam.set_xlabel("Epoch")
ax_gn_adam.semilogy(eps_a, smooth(ft_adam.h_gnorm, SMOOTH), color="tab:blue", lw=1.4)
ax_gn_adam.set_title("Adam — grad norm");   ax_gn_adam.set_xlabel("Epoch")

# L-BFGS
ep_l  = np.arange(len(ft_lbfgs.h_loss))
eps_l = np.arange(SMOOTH - 1, len(ft_lbfgs.h_loss))
ax_loss_lbfgs.plot(eps_l if len(eps_l) < SMOOTH else eps_l,
                   smooth(ft_lbfgs.h_loss,  SMOOTH) if len(ft_lbfgs.h_loss) >= SMOOTH else ft_lbfgs.h_loss,
                   color="tab:orange", lw=1.4)
ax_loss_lbfgs.axhline(0, color="gray", ls=":", lw=0.8)
ax_loss_lbfgs.set_title("L-BFGS — ZQE loss"); ax_loss_lbfgs.set_xlabel("Outer iter")
ax_gn_lbfgs.semilogy(ep_l, ft_lbfgs.h_gnorm, color="tab:orange", lw=1.4)
ax_gn_lbfgs.set_title("L-BFGS — grad norm");  ax_gn_lbfgs.set_xlabel("Outer iter")

plt.tight_layout(); plt.show()

In [ ]:
# ── Rotated loadings scatter ─────────────────────────────────────────────────
W_true_np = g0m.wz.detach().cpu().numpy()

def _aligned_w(W_t, W_h):
    return W_h @ _best_rotation(W_t, W_h)

plot_methods = [
    ("ZQE Adam",       "tab:blue",   ft_adam.model.wz.detach().cpu().numpy()),
    ("ZQE L-BFGS EM",  "tab:orange", ft_lbfgs.model.wz.detach().cpu().numpy()),
]
if W_hat_r is not None:
    plot_methods.append(("R gllvm VA", "tab:green", W_hat_r))

ncols = len(plot_methods)
fig_w, axes_w = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4), squeeze=False)
fig_w.suptitle("Estimated vs true loadings $W_z$ (Procrustes-rotated)", fontsize=11)

for ax, (label, col, W_hat) in zip(axes_w[0], plot_methods):
    W_rot = _aligned_w(W_true_np, W_hat)
    ax.scatter(W_true_np.ravel(), W_rot.ravel(),
               s=8, alpha=0.5, color=col, edgecolors="none")
    lim = max(np.abs(W_true_np).max(), np.abs(W_rot).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.9, alpha=0.6)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    R_str = [label for label, _, _ in plot_methods][plot_methods.index((label, col, W_hat))]
    err_val = {"ZQE Adam": err_adam, "ZQE L-BFGS EM": err_lbfgs, "R gllvm VA": err_r}.get(label, float("nan"))
    ax.set_title(f"{label}\nProcrustes={err_val:.4f}", fontsize=9)
    ax.set_xlabel("True $w_{ij}$", fontsize=8)
axes_w[0, 0].set_ylabel(r"Est. $\hat{w}_{ij}$", fontsize=8)

plt.tight_layout(); plt.show()